# Reranker Performance Evaluation

This notebook evaluates the impact of CrossEncoder reranking on search quality and latency.

In [ ]:
import sys
sys.path.insert(0, '..')

import time
import pandas as pd
from src.data_loader import load_articles
from src.rag_loader import HybridRAG

In [ ]:
# Load articles
articles = load_articles()
print(f"Loaded {len(articles)} articles")

In [ ]:
# Initialize RAG with and without reranker
print("Initializing RAG with reranker...")
rag_with_reranker = HybridRAG(articles, use_reranker=True)
rag_with_reranker.populate_database()

print("Initializing RAG without reranker...")
rag_without_reranker = HybridRAG(articles, use_reranker=False)
rag_without_reranker.populate_database()

print("✓ Both RAG systems initialized")

## Test Queries

A mix of specific and broad queries to evaluate search quality.

In [ ]:
TEST_QUERIES = [
    "modern minimalist bedroom design",
    "cozy living room with warm colors",
    "pink dining room ideas",
    "navy blue accent walls",
    "bohemian style decor",
    "small space kitchen solutions",
    "luxury master bedroom",
    "natural wood furniture",
    "coastal beach house interior",
    "industrial loft design"
]

## 1. Latency Comparison

In [ ]:
def measure_latency(rag, queries, top_k=5, runs=3):
    """Measure average latency for search queries."""
    latencies = []
    
    for query in queries:
        query_times = []
        for _ in range(runs):
            start = time.perf_counter()
            _ = rag.hybrid_search(query, top_k=top_k)
            elapsed = (time.perf_counter() - start) * 1000  # ms
            query_times.append(elapsed)
        latencies.append(sum(query_times) / runs)
    
    return latencies

print("Measuring latency (this may take a moment)...")
latency_with = measure_latency(rag_with_reranker, TEST_QUERIES)
latency_without = measure_latency(rag_without_reranker, TEST_QUERIES)

latency_df = pd.DataFrame({
    'Query': TEST_QUERIES,
    'With Reranker (ms)': [f"{l:.1f}" for l in latency_with],
    'Without Reranker (ms)': [f"{l:.1f}" for l in latency_without],
    'Overhead (ms)': [f"{w-wo:.1f}" for w, wo in zip(latency_with, latency_without)]
})

print(f"\n📊 Latency Results:")
print(f"   Avg with reranker:    {sum(latency_with)/len(latency_with):.1f} ms")
print(f"   Avg without reranker: {sum(latency_without)/len(latency_without):.1f} ms")
print(f"   Avg overhead:         {sum(latency_with)/len(latency_with) - sum(latency_without)/len(latency_without):.1f} ms")

latency_df

## 2. Search Results Comparison

In [ ]:
def compare_results(query, top_k=5):
    """Compare search results with and without reranker."""
    results_with = rag_with_reranker.hybrid_search(query, top_k=top_k)
    results_without = rag_without_reranker.hybrid_search(query, top_k=top_k)
    
    print(f"\n🔍 Query: \"{query}\"")
    print("=" * 60)
    
    print("\n✅ WITH Reranker:")
    for i, r in enumerate(results_with, 1):
        score = r.get('rerank_score', r.get('relevance_score', 0))
        print(f"  {i}. {r['title'][:50]} (score: {score:.3f})")
    
    print("\n❌ WITHOUT Reranker:")
    for i, r in enumerate(results_without, 1):
        score = r.get('hybrid_score', r.get('relevance_score', 0))
        print(f"  {i}. {r['title'][:50]} (score: {score:.1f})")
    
    # Check ranking changes
    titles_with = [r['title'] for r in results_with]
    titles_without = [r['title'] for r in results_without]
    
    if titles_with[0] != titles_without[0]:
        print(f"\n⚡ TOP RESULT CHANGED!")
    
    return results_with, results_without

In [ ]:
# Compare a few queries
for query in TEST_QUERIES[:5]:
    compare_results(query, top_k=3)

## 3. Ranking Change Analysis

In [ ]:
def analyze_ranking_changes(queries, top_k=5):
    """Analyze how often reranker changes the top results."""
    stats = {
        'top1_changed': 0,
        'top3_changed': 0,
        'new_in_top5': 0,
        'total_queries': len(queries)
    }
    
    for query in queries:
        with_rr = rag_with_reranker.hybrid_search(query, top_k=top_k)
        without_rr = rag_without_reranker.hybrid_search(query, top_k=top_k)
        
        titles_with = [r['title'] for r in with_rr]
        titles_without = [r['title'] for r in without_rr]
        
        # Top 1 changed?
        if titles_with and titles_without and titles_with[0] != titles_without[0]:
            stats['top1_changed'] += 1
        
        # Top 3 order changed?
        if titles_with[:3] != titles_without[:3]:
            stats['top3_changed'] += 1
        
        # New articles in top 5?
        new_articles = set(titles_with) - set(titles_without)
        if new_articles:
            stats['new_in_top5'] += 1
    
    return stats

stats = analyze_ranking_changes(TEST_QUERIES)

print("📈 Ranking Change Analysis:")
print(f"   Top-1 result changed: {stats['top1_changed']}/{stats['total_queries']} ({100*stats['top1_changed']/stats['total_queries']:.0f}%)")
print(f"   Top-3 order changed:  {stats['top3_changed']}/{stats['total_queries']} ({100*stats['top3_changed']/stats['total_queries']:.0f}%)")
print(f"   New articles in top-5: {stats['new_in_top5']}/{stats['total_queries']} ({100*stats['new_in_top5']/stats['total_queries']:.0f}%)")

## 4. Summary

In [ ]:
avg_overhead = sum(latency_with)/len(latency_with) - sum(latency_without)/len(latency_without)

print("=" * 60)
print("📋 RERANKER EVALUATION SUMMARY")
print("=" * 60)
print(f"\n⏱️  Latency overhead: ~{avg_overhead:.0f}ms per query")
print(f"🔄 Top-1 changes:    {stats['top1_changed']}/{stats['total_queries']} queries")
print(f"📊 Top-3 reordered:  {stats['top3_changed']}/{stats['total_queries']} queries")
print(f"\n💡 Recommendation:")
if avg_overhead < 100 and stats['top1_changed'] > 2:
    print("   ✅ Reranker provides meaningful improvements with acceptable latency.")
elif avg_overhead > 200:
    print("   ⚠️  Consider if latency overhead is acceptable for your use case.")
else:
    print("   ℹ️  Results are mixed. Review the comparison above for specific queries.")